In [1]:
# ============================================================
# CELL 1 — SETUP + LOAD DATA
# Loads unified train set, fixed test questions and
# already stored Top-5 RAG retrievals from Drive
# ============================================================

!pip install -q -U \
    unsloth unsloth_zoo transformers trl datasets \
    accelerate peft bitsandbytes \
    sacrebleu rapidfuzz bert-score==0.3.13

from google.colab import drive
drive.mount("/content/drive")

import os, json, re, gc, random, unicodedata
import numpy as np
import pandas as pd
import torch
from pathlib import Path

from datasets import load_dataset
from unsloth import FastLanguageModel, is_bf16_supported
from trl import SFTTrainer, SFTConfig

# ------------------------------------------------------------
# Locate project folder
# ------------------------------------------------------------

possible = [
    Path("/content/drive/MyDrive/Govt_Chatbot"),
    Path("/content/drive/MyDrive/Govt_Chatbots")
]

BASE = None

for p in possible:
    if (p / "RAG" / "test_questions.csv").exists():
        BASE = p
        break

if BASE is None:
    raise FileNotFoundError("Govt_Chatbot project not found.")

RAG_DIR = BASE / "RAG"

OUT = BASE / "Mistral" / "RAG+Fine-Tuned"
MODEL_DIR = OUT / "model_adapter"
CHECKPOINT_DIR = OUT / "checkpoints"

OUT.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Find unified training set created earlier
# ------------------------------------------------------------

train_candidates = [
    BASE / "Llama" / "Fine-Tuned" / "unified_train_used.json",
    BASE / "Llama" / "Fine-Tuned" / "unified_training_used.json",
    RAG_DIR / "unified_train.json"
]

TRAIN_PATH = None

for p in train_candidates:
    if p.exists():
        TRAIN_PATH = p
        break

if TRAIN_PATH is None:
    raise FileNotFoundError("Unified training JSON not found.")


# ------------------------------------------------------------
# Load training set
# ------------------------------------------------------------

raw_dataset = load_dataset(
    "json",
    data_files=str(TRAIN_PATH),
    split="train"
)

required = {"instruction", "output"}

missing = required - set(raw_dataset.column_names)

assert not missing, f"Missing columns: {missing}"


# ------------------------------------------------------------
# Load fixed test questions
# ------------------------------------------------------------

tests = pd.read_csv(
    RAG_DIR / "test_questions.csv"
).fillna("")

assert "question" in tests.columns
assert "gold" in tests.columns
assert (tests["gold"].astype(str).str.strip() != "").all()


# ------------------------------------------------------------
# Load previously stored retrievals
# ------------------------------------------------------------

with open(
    RAG_DIR / "retrievals.json",
    encoding="utf-8"
) as f:
    retrievals = json.load(f)

assert len(tests) == len(retrievals), \
    "Test/retrieval count mismatch."


# Save exact inputs used
tests.to_csv(
    OUT / "test_questions_used.csv",
    index=False,
    encoding="utf-8-sig"
)

with open(
    OUT / "retrievals_used.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        retrievals,
        f,
        ensure_ascii=False,
        indent=2
    )


print("Training file:", TRAIN_PATH)
print("Training QA:", len(raw_dataset))
print("Test QA:", len(tests))
print("Retrieval sets:", len(retrievals))
print("Output:", OUT)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.3/82.3 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 126.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 124.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3

Generating train split: 0 examples [00:00, ? examples/s]

Training file: /content/drive/MyDrive/Govt_Chatbot/Llama/Fine-Tuned/unified_train_used.json
Training QA: 938
Test QA: 248
Retrieval sets: 248
Output: /content/drive/MyDrive/Govt_Chatbot/Mistral/RAG+Fine-Tuned


In [2]:
# ============================================================
# CELL 2 — MISTRAL QLORA FINE-TUNING
# Same configuration as supplied fine-tune notebook
# Saves LoRA adapter to Drive
# ============================================================

SEED = 3407
MAX_SEQ_LENGTH = 1024

BASE_MODEL = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


# ------------------------------------------------------------
# Load Mistral in 4-bit
# ------------------------------------------------------------

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


# ------------------------------------------------------------
# Same LoRA configuration as friend's notebook
# ------------------------------------------------------------

model = FastLanguageModel.get_peft_model(
    model,

    r=16,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],

    lora_alpha=16,
    lora_dropout=0,
    bias="none",

    use_gradient_checkpointing="unsloth",
    random_state=SEED
)

model.print_trainable_parameters()


# ------------------------------------------------------------
# Same system prompt as friend's notebook
# ------------------------------------------------------------

SYSTEM_PROMPT = (
    "তুমি বাংলাদেশ সরকারের সরকারি সেবা সম্পর্কিত একজন সহায়ক সহকারী। "
    "প্রশ্নের সরাসরি, সংক্ষিপ্ত এবং নির্ভুল উত্তর বাংলায় দাও। "
    "শুধুমাত্র নির্ভরযোগ্য তথ্য দাও। "
    "যদি তথ্য জানা না থাকে, বলবে 'আমি জানি না'।"
)


# ------------------------------------------------------------
# Format with Mistral's native chat template
# ------------------------------------------------------------

def formatting(example):

    instruction = str(
        example.get("instruction", "") or ""
    ).strip()

    input_text = str(
        example.get("input", "") or ""
    ).strip()

    answer = str(
        example.get("output", "") or ""
    ).strip()

    user_text = instruction

    if input_text:
        user_text += "\n" + input_text

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_text
        },
        {
            "role": "assistant",
            "content": answer
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    if tokenizer.eos_token and not text.endswith(tokenizer.eos_token):
        text += tokenizer.eos_token

    return {"text": text}


formatted_dataset = raw_dataset.map(
    formatting,
    remove_columns=raw_dataset.column_names
)

print("Training examples:", len(formatted_dataset))
print("\nSample:\n")
print(formatted_dataset[0]["text"][:800])


# ------------------------------------------------------------
# Exact training settings from friend's notebook
# ------------------------------------------------------------

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=formatted_dataset,

    args=SFTConfig(

        output_dir=str(CHECKPOINT_DIR),

        # Dataset
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,
        packing=True,
        dataset_num_proc=2,

        # Training
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,

        # Optimizer
        learning_rate=2e-4,
        optim="adamw_8bit",
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        weight_decay=0.01,

        # Precision
        fp16=not is_bf16_supported(),
        bf16=is_bf16_supported(),

        # Stability
        max_grad_norm=1.0,

        # Logging / saving
        logging_steps=10,
        save_strategy="epoch",
        save_total_limit=2,

        report_to="none",
        seed=SEED
    )
)


print("\nStarting fine-tuning...")

trainer_stats = trainer.train()


# ------------------------------------------------------------
# Save LoRA adapter + tokenizer
# ------------------------------------------------------------

model.save_pretrained(
    str(MODEL_DIR)
)

tokenizer.save_pretrained(
    str(MODEL_DIR)
)


# Save training history
pd.DataFrame(
    trainer.state.log_history
).to_csv(
    OUT / "training_history.csv",
    index=False
)


# Save reproducibility config
run_config = {

    "base_model": BASE_MODEL,
    "method": "4-bit QLoRA with Unsloth",

    "training_examples": len(raw_dataset),

    "max_seq_length": 1024,

    "lora_r": 16,
    "lora_alpha": 16,
    "lora_dropout": 0,

    "target_modules": [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],

    "epochs": 3,

    "batch_size": 2,
    "gradient_accumulation_steps": 4,

    "learning_rate": 2e-4,

    "optimizer": "adamw_8bit",
    "scheduler": "cosine",

    "warmup_ratio": 0.03,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,

    "packing": True,
    "seed": 3407
}


with open(
    OUT / "training_config.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        run_config,
        f,
        ensure_ascii=False,
        indent=2
    )


print("\nFine-tuning complete.")
print("Adapter saved:", MODEL_DIR)

==((====))==  Unsloth 2026.9.2: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth 2026.9.2 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


trainable params: 41,943,040 || all params: 7,289,966,592 || trainable%: 0.5754


Map:   0%|          | 0/938 [00:00<?, ? examples/s]

...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 194 | Num Epochs = 3 | Total steps = 75
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,289,966,592 (0.58% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
10,1.100259
20,0.716624
30,0.567704
40,0.448228
50,0.417582
60,0.330843
70,0.321710


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Govt_Chatbot/Mistral/RAG+Fine-Tuned/checkpoints/checkpoint-25/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/Govt_Chatbot/Mistral/RAG+Fine-Tuned/checkpoints/checkpoint-25.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Govt_Chatbot/Mistral/RAG+Fine-Tuned/checkpoints/checkpoint-50/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/Govt_Chatbot/Mistral/RAG+Fine-Tuned/checkpoints/checkpoint-50.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Govt_Chatbot/Mistral/RAG+Fine-Tuned/checkpoints/checkpoint-75/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in /content/drive/MyDrive/Govt_Chatbot/Mistral/RAG+Fine-Tuned/checkpoints/checkpoint-75.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/Govt_Chatbo


Fine-tuning complete.
Adapter saved: /content/drive/MyDrive/Govt_Chatbot/Mistral/RAG+Fine-Tuned/model_adapter


In [3]:
# ============================================================
# CELL 3 — RAG + FINE-TUNED MISTRAL INFERENCE
# Stored Top-5 contexts -> Fine-Tuned Mistral
# max_new_tokens = 500
# ============================================================

from tqdm.auto import tqdm

MAX_NEW_TOKENS = 500

FastLanguageModel.for_inference(model)

tokenizer.padding_side = "left"


# ------------------------------------------------------------
# Generate one RAG answer
# ------------------------------------------------------------

def generate_answer(question, docs):

    # Combine the same stored top retrievals
    context = "\n\n".join([
        f"[Context {i}]\n"
        f"শিরোনাম: {doc.get('title', '')}\n"
        f"তথ্য: {doc.get('text', '')}"
        for i, doc in enumerate(docs, 1)
    ])

    # Same RAG structure recommended in friend's notebook
    user_text = (
        f"প্রশ্ন: {question}\n\n"
        "প্রাসঙ্গিক তথ্য:\n"
        f"{context}\n\n"
        "উপরের প্রাসঙ্গিক তথ্য ব্যবহার করে প্রশ্নের "
        "সরাসরি ও সংক্ষিপ্ত উত্তর বাংলায় দাও।"
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_text
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=7000
    ).to("cuda")


    with torch.no_grad():

        outputs = model.generate(
            **inputs,

            max_new_tokens=MAX_NEW_TOKENS,

            do_sample=False,

            use_cache=True,

            repetition_penalty=1.05,

            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )


    generated = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()


    truncated = (
        len(generated) >= MAX_NEW_TOKENS
        and generated[-1].item() != tokenizer.eos_token_id
    )

    return answer, truncated


# ============================================================
# RESUME SUPPORT
# ============================================================

PARTIAL = OUT / "predictions_partial.csv"

done = {}

if PARTIAL.exists():

    old = pd.read_csv(
        PARTIAL
    ).fillna("")

    for _, row in old.iterrows():

        done[
            (
                str(row["domain"]),
                str(row["id"])
            )
        ] = row.to_dict()


print("Already completed:", len(done))


# ============================================================
# INFERENCE
# ============================================================

predictions = []

for i, row in tqdm(
    tests.iterrows(),
    total=len(tests),
    desc="Mistral RAG + Fine-Tuned"
):

    key = (
        str(row["domain"]),
        str(row["id"])
    )

    if key in done:

        result = done[key]

    else:

        answer, truncated = generate_answer(
            row["question"],
            retrievals[i]
        )

        result = {
            "id": str(row["id"]),
            "domain": str(row["domain"]),
            "question": row["question"],
            "gold": row["gold"],
            "prediction": answer,
            "truncated": truncated
        }

        done[key] = result


    predictions.append(result)


    # Save after every prediction
    pd.DataFrame(
        predictions
    ).to_csv(
        PARTIAL,
        index=False,
        encoding="utf-8-sig"
    )


# ------------------------------------------------------------
# Final predictions
# ------------------------------------------------------------

pred_df = pd.DataFrame(predictions)

pred_df.to_csv(
    OUT / "predictions.csv",
    index=False,
    encoding="utf-8-sig"
)


truncated_count = (
    pred_df["truncated"]
    .astype(str)
    .str.lower()
    .isin(["true", "1", "yes"])
    .sum()
)


print("\nCompleted:", len(pred_df))
print("Truncated outputs:", truncated_count)

print("Saved:")
print(OUT / "predictions.csv")


# Free GPU before BERTScore
del model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

Already completed: 0


Mistral RAG + Fine-Tuned:   0%|          | 0/248 [00:00<?, ?it/s]

Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=500) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene


Completed: 248
Truncated outputs: 1
Saved:
/content/drive/MyDrive/Govt_Chatbot/Mistral/RAG+Fine-Tuned/predictions.csv


In [4]:
# ============================================================
# CELL 4 — FINAL EVALUATION
# EM, Fuzzy, BLEU, ROUGE-1/2/L, Token F1,
# BERT Precision/Recall/F1 + Truncated Outputs
# ============================================================

from collections import Counter

from rapidfuzz import fuzz
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score


df = pd.read_csv(
    OUT / "predictions.csv"
).fillna("")

assert (
    df["gold"]
    .astype(str)
    .str.strip()
    != ""
).all(), "Blank gold answers found."


# ------------------------------------------------------------
# Normalization
# ------------------------------------------------------------

BN_TO_EN = str.maketrans(
    "০১২৩৪৫৬৭৮৯",
    "0123456789"
)


def normalize(text):

    text = unicodedata.normalize(
        "NFKC",
        str(text)
    )

    text = text.translate(
        BN_TO_EN
    ).lower()

    text = re.sub(
        r"[^\u0980-\u09FFA-Za-z0-9]+",
        " ",
        text
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def tokens(text):
    return normalize(text).split()


# ------------------------------------------------------------
# Exact Match
# ------------------------------------------------------------

def exact_match(pred, gold):

    return float(
        normalize(pred)
        ==
        normalize(gold)
    )


# ------------------------------------------------------------
# Token F1
# ------------------------------------------------------------

def token_f1(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p or not g:
        return 0.0

    overlap = sum(
        (Counter(p) & Counter(g)).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(p)
    recall = overlap / len(g)

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ------------------------------------------------------------
# ROUGE-N
# ------------------------------------------------------------

def rouge_n(pred, gold, n):

    p = tokens(pred)
    g = tokens(gold)

    if len(p) < n or len(g) < n:
        return 0.0

    pn = Counter(
        tuple(p[i:i+n])
        for i in range(len(p)-n+1)
    )

    gn = Counter(
        tuple(g[i:i+n])
        for i in range(len(g)-n+1)
    )

    overlap = sum(
        (pn & gn).values()
    )

    if overlap == 0:
        return 0.0

    precision = overlap / sum(pn.values())
    recall = overlap / sum(gn.values())

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ------------------------------------------------------------
# ROUGE-L
# ------------------------------------------------------------

def rouge_l(pred, gold):

    p = tokens(pred)
    g = tokens(gold)

    if not p or not g:
        return 0.0

    dp = [0] * (len(g) + 1)

    for x in p:

        new = [0]

        for j, y in enumerate(g, 1):

            if x == y:
                new.append(
                    dp[j-1] + 1
                )

            else:
                new.append(
                    max(
                        dp[j],
                        new[-1]
                    )
                )

        dp = new

    lcs = dp[-1]

    precision = lcs / len(p)
    recall = lcs / len(g)

    if precision + recall == 0:
        return 0.0

    return (
        2 * precision * recall
        /
        (precision + recall)
    )


# ============================================================
# ROW METRICS
# ============================================================

df["Exact Match"] = [
    exact_match(p, g)
    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["Fuzzy Match"] = [
    fuzz.token_set_ratio(
        normalize(p),
        normalize(g)
    ) / 100

    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["Token F1"] = [
    token_f1(p, g)

    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["ROUGE-1"] = [
    rouge_n(p, g, 1)

    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["ROUGE-2"] = [
    rouge_n(p, g, 2)

    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]

df["ROUGE-L"] = [
    rouge_l(p, g)

    for p, g in zip(
        df["prediction"],
        df["gold"]
    )
]


# ============================================================
# CORPUS BLEU
# ============================================================

bleu = BLEU(
    tokenize="none",
    smooth_method="exp",
    effective_order=True
)

pred_bleu = [
    " ".join(tokens(x))
    for x in df["prediction"]
]

gold_bleu = [
    " ".join(tokens(x))
    for x in df["gold"]
]

corpus_bleu = (
    bleu.corpus_score(
        pred_bleu,
        [gold_bleu]
    ).score
    / 100
)


# ============================================================
# BERTSCORE
# ============================================================

print("Calculating BERTScore...")

P, R, F1 = bert_score(
    df["prediction"].astype(str).tolist(),
    df["gold"].astype(str).tolist(),

    model_type="bert-base-multilingual-cased",

    batch_size=4,
    device="cpu",

    idf=False,
    rescale_with_baseline=False,

    verbose=True
)

df["BERT Precision"] = P.cpu().numpy()
df["BERT Recall"] = R.cpu().numpy()
df["BERT F1"] = F1.cpu().numpy()


# ============================================================
# TRUNCATED OUTPUTS
# ============================================================

truncated_count = (
    df["truncated"]
    .astype(str)
    .str.lower()
    .isin(["true", "1", "yes"])
    .sum()
)


# ============================================================
# FINAL RESULT
# ============================================================

result = pd.DataFrame({

    "metric": [
        "Exact Match",
        "Fuzzy Match",
        "Corpus BLEU",
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "Token F1",
        "BERT Precision",
        "BERT Recall",
        "BERT F1",
        "Truncated Outputs"
    ],

    "score": [
        df["Exact Match"].mean(),
        df["Fuzzy Match"].mean(),
        corpus_bleu,
        df["ROUGE-1"].mean(),
        df["ROUGE-2"].mean(),
        df["ROUGE-L"].mean(),
        df["Token F1"].mean(),
        df["BERT Precision"].mean(),
        df["BERT Recall"].mean(),
        df["BERT F1"].mean(),
        int(truncated_count)
    ]
})


# Save detailed predictions
df.to_csv(
    OUT / "predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

# Save final metrics
result.to_csv(
    OUT / "result.csv",
    index=False,
    encoding="utf-8-sig"
)

display(result)


print("\nSaved artifacts:")
print(OUT / "model_adapter")
print(OUT / "training_config.json")
print(OUT / "training_history.csv")
print(OUT / "retrievals_used.json")
print(OUT / "test_questions_used.csv")
print(OUT / "predictions_partial.csv")
print(OUT / "predictions.csv")
print(OUT / "result.csv")

Calculating BERTScore...


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

...

computing greedy matching.


  0%|          | 0/62 [00:00<?, ?it/s]

done in 67.67 seconds, 3.66 sentences/sec


,metric,score
0,Exact Match,0.169355
1,Fuzzy Match,0.867978
2,Corpus BLEU,0.506216
3,ROUGE-1,0.651847
4,ROUGE-2,0.561368
5,ROUGE-L,0.630483
6,Token F1,0.651847
7,BERT Precision,0.871338
8,BERT Recall,0.877769
9,BERT F1,0.872702



Saved artifacts:
/content/drive/MyDrive/Govt_Chatbot/Mistral/RAG+Fine-Tuned/model_adapter
/content/drive/MyDrive/Govt_Chatbot/Mistral/RAG+Fine-Tuned/training_config.json
/content/drive/MyDrive/Govt_Chatbot/Mistral/RAG+Fine-Tuned/training_history.csv
/content/drive/MyDrive/Govt_Chatbot/Mistral/RAG+Fine-Tuned/retrievals_used.json
/content/drive/MyDrive/Govt_Chatbot/Mistral/RAG+Fine-Tuned/test_questions_used.csv
/content/drive/MyDrive/Govt_Chatbot/Mistral/RAG+Fine-Tuned/predictions_partial.csv
/content/drive/MyDrive/Govt_Chatbot/Mistral/RAG+Fine-Tuned/predictions.csv
/content/drive/MyDrive/Govt_Chatbot/Mistral/RAG+Fine-Tuned/result.csv
